In [22]:
cd /Users/amedin/Research/SummaBMI/ngen/extern/summa/summa/test_ngen

/Users/amedin/Research/SummaBMI/ngen/extern/summa/summa/test_ngen


In [107]:
%%bash
python3 scripts/tile_attributes_by_hru.py gauge_01073000/settings/SUMMA/attributes.nc ../../../../data/gauge_01073000/forcing.nc attrib ../../../../data/gauge_01073000

Wrote tiled old_file to gauge_01073000/settings/SUMMA/attributes_tiled_by_hru.nc


In [150]:
%%bash
python3 scripts/tile_attributes_by_hru.py gauge_01073000/settings/SUMMA/trialParams_default.nc ../../../../data/gauge_01073000/forcing.nc param ../../../../data/gauge_01073000

Wrote tiled old_file to gauge_01073000/settings/SUMMA/trialParams_default_tiled_by_hru.nc


In [34]:
%%bash
python3 scripts/tile_attributes_by_hru.py gauge_01073000/settings/SUMMA/coldstate.nc ../../../../data/gauge_01073000/forcing.nc init ../../../../data/gauge_01073000

Wrote tiled old_file to gauge_01073000/settings/SUMMA/coldstate_tiled_by_hru.nc


In [5]:
# read netcdf files
import xarray as xr
import numpy as np

xr.open_dataset('../../../../../data/gauge_01073000/forcing.nc')

# make geojson from gpkg
import geopandas as gpd
#gdf = gpd.read_file('/Users/amedin/Research/SummaBMI/ngen/data/gauge_01073000/gauge_01073000.gpkg')
#gdf.to_file('../../../../../data/gauge_01073000/catchment.geojson', driver='GeoJSON')   

SyntaxError: unterminated string literal (detected at line 6) (4106930797.py, line 6)

In [35]:
ds = xr.open_dataset('gauge_01073000/settings/SUMMA/coldstate_tiled_by_hru.nc')
print(ds['mLayerVolFracLiq'].values)
print(ds['mLayerMatricHead'])

[[0.3 0.3 0.3 0.3 0.3]
 [0.3 0.3 0.3 0.3 0.3]
 [0.3 0.3 0.3 0.3 0.3]
 [0.3 0.3 0.3 0.3 0.3]
 [0.3 0.3 0.3 0.3 0.3]
 [0.3 0.3 0.3 0.3 0.3]
 [0.3 0.3 0.3 0.3 0.3]
 [0.3 0.3 0.3 0.3 0.3]]
<xarray.DataArray 'mLayerMatricHead' (midSoil: 8, hru: 5)> Size: 320B
[40 values with dtype=float64]
Dimensions without coordinates: midSoil, hru


In [ ]:
import re
import pandas as pd
import numpy as np
from io import StringIO

text = """191   BB      DRYSMC      F11     MAXSMC   REFSMC   SATPSI  SATDK       SATDW     WLTSMC  QTZ    '
1     2.79    0.010    -0.472   0.339   0.236   0.069  1.07E-6  0.608E-6   0.010  0.92 'SAND'
2     4.26    0.028    -1.044   0.421   0.383   0.036  1.41E-5  0.514E-5   0.028  0.82 'LOAMY SAND'
3     4.74    0.047    -0.569   0.434   0.383   0.141  5.23E-6  0.805E-5   0.047  0.60 'SANDY LOAM'
4     5.33    0.084     0.162   0.476   0.360   0.759  2.81E-6  0.239E-4   0.084  0.25 'SILT LOAM'
5     5.33    0.084     0.162   0.476   0.383   0.759  2.81E-6  0.239E-4   0.084  0.10 'SILT'
6     5.25    0.066    -0.327   0.439   0.329   0.355  3.38E-6  0.143E-4   0.066  0.40 'LOAM'
7     6.66    0.067    -1.491   0.404   0.314   0.135  4.45E-6  0.990E-5   0.067  0.60 'SANDY CLAY LOAM'
8     8.72    0.120    -1.118   0.464   0.387   0.617  2.04E-6  0.237E-4   0.120  0.10 'SILTY CLAY LOAM'
9     8.17    0.103    -1.297   0.465   0.382   0.263  2.45E-6  0.113E-4   0.103  0.35 'CLAY LOAM'
10   10.73    0.100    -3.209   0.406   0.338   0.098  7.22E-6  0.187E-4   0.100  0.52 'SANDY CLAY'
11   10.39    0.126    -1.916   0.468   0.404   0.324  1.34E-6  0.964E-5   0.126  0.10 'SILTY CLAY'
12   11.55    0.138    -2.138   0.468   0.412   0.468  9.74E-7  0.112E-4   0.138  0.25 'CLAY'
13    5.25    0.066    -0.327   0.439   0.329   0.355  3.38E-6  0.143E-4   0.066  0.05 'ORGANIC MATERIAL'
14     0.0      0.0       0.0     1.0     0.0     0.0      0.0       0.0     0.0  0.60 'WATER'
15    2.79    0.006    -1.111    0.20    0.17   0.069  1.41E-4  0.136E-3   0.006  0.07 'BEDROCK'
16    4.26    0.028    -1.044   0.421   0.283   0.036  1.41E-5  0.514E-5   0.028  0.25 'OTHER(land-ice)'
17   11.55    0.030   -10.472   0.468   0.454   0.468  9.74E-7  0.112E-4   0.030  0.60 'PLAYA'
18    2.79    0.006    -0.472   0.200    0.17   0.069  1.41E-4  0.136E-3   0.006  0.52 'LAVA'
19    2.79     0.01    -0.472   0.339   0.236   0.069  1.07E-6  0.608E-6    0.01  0.92 'WHITE SAND'
"""

#text ="""191   BB      DRYSMC       HC     MAXSMC   REFSMC   SATPSI  SATDK       SATDW     WLTSMC  QTZ    '
#1     4.05    0.045      1.47   0.395   0.236   0.121  1.76E-4  0.608E-6   0.068  0.92 'SAND'
#2     4.38    0.057      1.41   0.410   0.383   0.090  1.56E-4  0.514E-5   0.075  0.82 'LOAMY SAND'
#3     4.90    0.065      1.34   0.435   0.383   0.218  3.47E-5  0.805E-5   0.114  0.60 'SANDY LOAM'
#4     5.30    0.067      1.27   0.485   0.360   0.786  7.20E-6  0.239E-4   0.179  0.25 'SILT LOAM'
#5     5.30    0.034      1.27   0.485   0.383   0.786  7.20E-6  0.239E-4   0.179  0.10 'SILT'
#6     5.39    0.078      1.21   0.451   0.329   0.478  6.95E-6  0.143E-4   0.155  0.40 'LOAM'
#7     7.12    0.100      1.18   0.420   0.314   0.299  6.30E-6  0.990E-5   0.175  0.60 'SANDY CLAY LOAM'
#8     7.75    0.089      1.32   0.477   0.387   0.356  1.70E-6  0.237E-4   0.218  0.10 'SILTY CLAY LOAM'
#9     8.52    0.095      1.23   0.476   0.382   0.630  2.45E-6  0.113E-4   0.250  0.35 'CLAY LOAM'
#10   10.40    0.100      1.18   0.426   0.338   0.153  2.17E-6  0.187E-4   0.219  0.52 'SANDY CLAY'
#11   10.40    0.070      1.15   0.492   0.404   0.490  1.03E-6  0.964E-5   0.283  0.10 'SILTY CLAY'
#12   11.40    0.068      1.09   0.482   0.412   0.405  1.28E-6  0.112E-4   0.286  0.25 'CLAY'
#13    5.39    0.078      1.21   0.451   0.329   0.478  6.95E-6  0.143E-4   0.155  0.05 'ORGANIC MATERIAL'
#14     0.0      0.0      4.18   1.0     1.0     0.0      0.0       0.0     0.0    0.00 'WATER'
#15    4.05    0.004      2.03   0.200   0.17   0.121  1.41E-4  0.136E-3   0.006  0.60 'BEDROCK'
#16    4.90    0.065      2.10   0.435   0.283   0.218  3.47E-5  0.514E-5   0.114  0.05 'OTHER(land-ice)'
#17   11.40    0.030      1.41   0.468   0.454   0.468  9.74E-7  0.112E-4   0.030  0.60 'PLAYA'
#18    4.05    0.006      1.41   0.200   0.17   0.069  1.41E-4  0.136E-3   0.060  0.52 'LAVA'
#19    4.05     0.01      1.47   0.339   0.236   0.069  1.76E-4  0.608E-6   0.060  0.92 'WHITE SAND'
#"""

# column names in order (adjust if your table differs)
num_cols = ["idx","BB","DRYSMC","HC",
            "MAXSMC","REFSMC","SATPSI","SATDK","SATDW","WLTSMC","QTZ"]

number_re = r'[-+]?\d*\.?\d+(?:[Ee][+-]?\d+)?'

rows = []
names = []
for line in text.splitlines():
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    nums = re.findall(number_re, line)
    if len(nums) < len(num_cols):
        continue
    nums = nums[:len(num_cols)]
    nums = [float(x) for x in nums]
    name_m = re.search(r"'([^']+)'", line)
    name = name_m.group(1).strip() if name_m else ""
    rows.append(nums)
    names.append(name)

df = pd.DataFrame(rows, columns=num_cols)
df["name"] = names
df = df.set_index("idx")
print(df[["SATPSI","BB"]])

df['p'] = 3 + 2* df['BB'] 
df['n_estimate'] = (1+df['BB'])/(df['BB'])
df['alpha_estimate'] = -1/(df['SATPSI']) * (df['p']+3)/(2*df['p']*(df['p']-1)) * (147.8+8.1*df['p']+0.092*df['p']**2) / (55.6+7.4*df['p']+df['p']**2)



print(df[["alpha_estimate", "n_estimate"]])

      SATPSI     BB
idx                
1.0    0.069   2.79
2.0    0.036   4.26
3.0    0.141   4.74
4.0    0.759   5.33
5.0    0.759   5.33
6.0    0.355   5.25
7.0    0.135   6.66
8.0    0.617   8.72
9.0    0.263   8.17
10.0   0.098  10.73
11.0   0.324  10.39
12.0   0.468  11.55
13.0   0.355   5.25
14.0   0.000   0.00
15.0   0.069   2.79
16.0   0.036   4.26
17.0   0.468  11.55
18.0   0.069   2.79
19.0   0.069   2.79
      alpha_estimate  n_estimate
idx                             
1.0        -1.500229    1.358423
2.0        -1.540947    1.234742
3.0        -0.332071    1.210970
4.0        -0.050953    1.187617
5.0        -0.050953    1.187617
6.0        -0.111688    1.190476
7.0        -0.196846    1.150150
8.0        -0.026923    1.114679
9.0        -0.070844    1.122399
10.0       -0.117257    1.093197
11.0       -0.037563    1.096246
12.0       -0.021529    1.086580
13.0       -0.111688    1.190476
14.0            -inf         inf
15.0       -1.500229    1.358423
16.0       -1.54094